In [194]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [2]:
X,y = load_diabetes(return_X_y=True)

In [65]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2 ,random_state=42)

In [195]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [196]:
class MeraRidgeSGDSimplified:
    def __init__(self,epoch,learning_rate,alpha):
        self.coef_ = None
        self.intercept_ = None
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
    
    def fit(self,X_train,y_train):
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        weights = np.insert(self.coef_,0,self.intercept_)
        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epoch):
            idx = np.random.randint(X_train.shape[0])
            penalty = self.alpha * weights.copy()
            penalty[0] = 0
            error = np.dot(X_train[idx],weights) - y_train[idx]
            der = np.dot(X_train[idx].T,error) + penalty
            weights = weights - self.learning_rate * der

        self.intercept_ = weights[0]
        self.coef_ = weights[1:]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_


__Second Version of the Formulae__

In [267]:
class MeraRidgeSGD:
    def __init__(self,epoch,learning_rate,alpha):
        self.coef_ = None
        self.intercept_ = None
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
    
    def fit(self,X_train,y_train):
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        weights = np.insert(self.coef_,0,self.intercept_)
        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epoch):
            idx = np.random.randint(X_train.shape[0])
            penalty = self.alpha * weights.copy()
            penalty[0] = 0
            der = np.outer(X_train[idx], X_train[idx]).dot(weights) - X_train[idx] * y_train[idx] + penalty
            weights = weights - self.learning_rate * der

        self.intercept_ = weights[0]
        self.coef_ = weights[1:]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

In [268]:
ridge = MeraRidgeSGD(epoch = 10000 , learning_rate= 0.01 , alpha = 0.5)
ridge.fit(X_train_scaled,y_train)
y_pred_cp = ridge.predict(X_test_scaled)

In [269]:
from sklearn.metrics import r2_score
from sklearn.linear_model import SGDRegressor

# My Simplified SGD
ridge_s = MeraRidgeSGDSimplified(epoch=10000, learning_rate=0.01, alpha=0.5)
ridge_s.fit(X_train_scaled, y_train)
y_pred_yours = ridge_s.predict(X_test_scaled)

# My SGD Variation
ridge = MeraRidgeSGD(epoch = 10000 , learning_rate= 0.01 , alpha = 0.5)
ridge.fit(X_train_scaled,y_train)
y_pred_cp = ridge.predict(X_test_scaled)

# Sklearn SGDRegressor (fair comparison)
sklearn_sgd = SGDRegressor(
    penalty='l2',
    alpha=0.5,
    learning_rate='constant',
    eta0=0.01,
    max_iter=10000,
    shuffle=True,
    fit_intercept=True
)
sklearn_sgd.fit(X_train_scaled, y_train)
y_pred_sklearn = sklearn_sgd.predict(X_test_scaled)


# R2 comparison
print("My SGD Simplified     → R2:", r2_score(y_test, y_pred_yours))
print("My SGD Complex        → R2:",r2_score(y_test,y_pred_cp))
print("Sklearn SGD  → R2:", r2_score(y_test, y_pred_sklearn))

# Coef comparison
print("\nMy coef Simplified  :", ridge_s.coef_)
print("\nMy coef Comlpex     :", ridge.coef_)
print("Sklearn coef:", sklearn_sgd.coef_)

print("\nMy intercept Simplified  :", ridge_s.intercept_)
print("\nMy intercept Complex     :", ridge.intercept_)
print("Sklearn intercept:", sklearn_sgd.intercept_)

My SGD Simplified     → R2: 0.4495369036917153
My SGD Complex        → R2: 0.4412477414183865
Sklearn SGD  → R2: 0.4084986276033079

My coef Simplified  : [  0.77228041  -2.47639239  24.05547217  11.78656319  -1.66081045
  -2.80174117 -10.38328171   8.71283263  16.74403511   6.02796287]

My coef Comlpex     : [  1.20347862  -9.47344886  18.04873425   9.97333199  -2.25983677
  -2.31740013 -12.25964422   8.94184267  15.86305282   7.08271741]
Sklearn coef: [-0.55445351 -9.1972338  21.21444498 10.20683018 -2.80588412 -4.38986316
 -5.32731438  3.93154367 11.67208821  3.00187885]

My intercept Simplified  : 158.4395540119634

My intercept Complex     : 144.9994664958607
Sklearn intercept: [148.1561407]
